# Baseline Submission Pipeline (BM25 + Medical Dictionary Query Expansion + Dynamic Window 2500 chars)
Rút kinh nghiệm từ Exp 11: Thay thế mô hình dịch NLLB (làm giảm recall) bằng việc mở rộng truy vấn (Query Expansion) qua từ điển đồng nghĩa y khoa (ICD-10/custom map). Vẫn giữ nguyên sliding window 2500 chars (Exp 09) để đảm bảo không gọt mất thông tin quan trọng ở đầu đoạn, đồng thời dung lượng zip <100MB.


In [117]:
!pip install -q bm25s pandas pyarrow numpy sentencepiece


In [118]:
import pandas as pd
import glob
import bm25s
import numpy as np
from collections import Counter

# --- HÀM TÌM CỬA SỔ ĐỘNG TỐI ƯU ---
def select_best_window(text, query_tokens, win_chars=2500, stride=800):
    if not text:
        return 0, 0
    if len(text) <= win_chars:
        return 0, len(text) # Nộp nguyên bài nếu ngắn hơn ngân sách
    
    # Sinh danh sách các cửa sổ trượt
    starts = list(range(0, max(len(text) - win_chars, 0) + 1, stride))
    if starts[-1] != len(text) - win_chars:
        starts.append(max(len(text) - win_chars, 0))
    wins = [(s, min(s + win_chars, len(text))) for s in starts]
    
    # Chấm điểm cực nhanh bằng việc đếm số từ khóa xuất hiện trong mỗi cửa sổ
    q_set = set([q.lower() for q in query_tokens if q.strip()])
    if not q_set:
        return 0, win_chars
        
    best_score = -1
    best_win = (0, win_chars)
    
    for s, e in wins:
        window_text = text[s:e].lower()
        # Đếm tần suất xuất hiện của các từ khóa query trong cửa sổ này
        score = sum(window_text.count(q) for q in q_set)
        if score > best_score:
            best_score = score
            best_win = (s, e)
            
    return best_win


In [119]:
# 1. Quét danh sách file
corpus_files = glob.glob('/kaggle/input/r2ai-stage-3-raw-corpus-text/**/*.parquet', recursive=True)
if not corpus_files:
    print("CẢNH BÁO: KHÔNG TÌM THẤY DỮ LIỆU.")

doc_ids = []
corpus_texts = []
print("1. Đang nạp toàn bộ 4.4 triệu bài báo vào RAM...")

for f in corpus_files:
    df = pd.read_parquet(f, columns=['id', 'text']).dropna(subset=['text'])
    doc_ids.extend(df['id'].astype(str).tolist())
    # LƯU TOÀN BỘ TEXT ĐỂ CẮT ĐỘNG, KHÔNG CẮT CỨNG TỪ ĐẦU NỮA
    corpus_texts.extend(df['text'].astype(str).tolist())

print(f"Tổng số bài báo: {len(doc_ids)}")

# 2. Load BM25 Index
print("2. Load BM25 Index từ Dataset...")
INDEX_DIR = "/kaggle/input/TEN-DATASET-CUA-BAN/bm25_index_dir"
retriever = bm25s.BM25.load(INDEX_DIR, mmap=True)
print("Hoàn tất Load Index! Đã sẵn sàng tìm kiếm.")


CẢNH BÁO: KHÔNG TÌM THẤY DỮ LIỆU.
1. Đang nạp toàn bộ 4.4 triệu bài báo vào RAM...
Tổng số bài báo: 0
2. Load BM25 Index từ Dataset...


FileNotFoundError: [Errno 2] No such file or directory: '\\kaggle\\input\\TEN-DATASET-CUA-BAN\\bm25_index_dir\\params.index.json'

In [ ]:
import json
import zipfile
import re

# Từ điển đồng nghĩa y khoa cơ bản (Medical Synonyms Dictionary)
# Có thể mở rộng bằng chuẩn ICD-10 theo ý tưởng của Exp 11.
MEDICAL_SYNONYMS = {
    "đái tháo đường": ["tiểu đường", "diabetes", "đtđ"],
    "tăng huyết áp": ["cao huyết áp", "hypertension", "tha"],
    "ung thư": ["cancer", "k", "khối u", "ác tính"],
    "viêm gan": ["hepatitis"],
    "nhồi máu cơ tim": ["đột quỵ tim", "heart attack", "myocardial infarction"],
    "đột quỵ": ["tai biến mạch máu não", "stroke"],
    "béo phì": ["thừa cân", "obesity"],
    "sốt xuất huyết": ["dengue"],
    "đau dạ dày": ["viêm loét dạ dày", "đau bao tử", "stomach ache"],
    "lao phổi": ["tuberculosis", "tb"]
}

def expand_query(query):
    query_lower = query.lower()
    expanded_terms = []
    for key, synonyms in MEDICAL_SYNONYMS.items():
        if key in query_lower:
            expanded_terms.extend(synonyms)
    if expanded_terms:
        # Mở rộng truy vấn gốc với các từ đồng nghĩa
        return query + " " + " ".join(expanded_terms)
    return query

df_queries = pd.read_parquet('/kaggle/input/TÊN-DATASET-CUỘC-THI/query.parquet')

results = []
print("4. Đang chạy tìm kiếm với Query Expansion (BM25 + Medical Dictionary)...")

for idx, row in df_queries.iterrows():
    q_id = int(row['id'])
    q_text_vi = str(row['query'])
    
    # 1. Mở rộng truy vấn bằng từ điển
    q_expanded = expand_query(q_text_vi)
    
    # 2. Tokenize query
    q_tokens = bm25s.tokenize([q_expanded])
    
    # 3. Truy hồi thẳng top 100 tài liệu
    doc_indices, _ = retriever.retrieve(q_tokens, k=100)
    
    top_k_indices = doc_indices[0]
    result_doc_ids = [int(doc_ids[i]) for i in top_k_indices]
    
    # --- TRÍCH XUẤT CỬA SỔ ĐỘNG ---
    # Tách từ khóa để đếm mật độ (bao gồm cả từ khóa gốc và mở rộng)
    q_tokens_raw = q_expanded.split()
    result_chunks = []
    
    for i in top_k_indices:
        full_text = corpus_texts[i]
        # Ép win_chars = 2500 để đảm bảo zip < 100MB (Theo Exp 09), không gọt đầu xén đuôi
        s, e = select_best_window(full_text, q_tokens_raw, win_chars=2500, stride=800)
        result_chunks.append({
            "doc_id": int(doc_ids[i]),
            "chunk_text": full_text[s:e]
        })
    
    results.append({
        "id": q_id,
        "relevant_docs": result_doc_ids,
        "relevant_chunks": result_chunks
    })
    
    if (idx + 1) % 10 == 0:
        print(f"Đã xử lý {idx + 1}/{len(df_queries)} truy vấn...\n")

with open('submission.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, separators=(',', ':'))
print("Đã tạo file submission.json thành công!\n")

with zipfile.ZipFile('submission.zip', 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    zf.write('submission.json')
print("🎉 Đã nén thành submission.zip siêu nhẹ. HÃY NỘP FILE NÀY THÔI!")
